In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("data/categories.csv")
df.head()
df = pd.read_csv("data/customers.csv")
df.head()
df = pd.read_csv("data/merchants.csv")
df.head()
df = pd.read_csv("data/transactions.csv")
df.head()

print(df.shape)
df.info()
df["amount"].describe()
print(df["amount"].describe())

# Q1

summary = df.groupby("category")["amount"].agg(total="sum", count="count")
summary["ratio"] = (summary["total"] / summary["total"].sum() * 100).round(1)
summary = summary.sort_values("total", ascending=False)
print(summary)

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

plt.figure(figsize=(10, 6))
plt.bar(summary.index, summary["ratio"])
plt.title("카테고리별 지출 비중(%)")
plt.xlabel("카테고리")
plt.ylabel("비중(%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("category_ratio.png")
plt.show()

# Q2    

THRESHOLD = 150000

big = df[df["amount"] > THRESHOLD]
print(len(big))
print(big["amount"].sum())
print(big.sort_values("amount", ascending=False).head(10))

top10 = big.sort_values("amount", ascending=False).head(10)
print(top10[["date", "category", "amount", "channel"]])

# (a) 카테고리별
big_cat = big.groupby("category")["amount"].agg(total="sum", count="count")
big_cat["ratio"] = (big_cat["total"] / big_cat["total"].sum() * 100).round(1)
big_cat = big_cat.sort_values("total", ascending=False)
print(big_cat)

# (b) 월별
big["month"] = big["date"].str[:7]
big_month = big.groupby("month")["amount"].agg(total="sum", count="count")
print(big_month)

# (c) 채널별
print(big["channel"].value_counts())

summary.to_csv("pandas_q1_category.csv", encoding="utf-8-sig")
big_cat.to_csv("pandas_q2_big_category.csv", encoding="utf-8-sig")
big_month.to_csv("pandas_q2_big_month.csv", encoding="utf-8-sig")

# 인사이트 숫자 검증
print(summary["total"].sum())                                                    # 전체 합계
print((summary["total"] / summary["count"]).round(0))                            # 카테고리별 건당 평균
print(round(summary["total"].head(3).sum() / summary["total"].sum() * 100, 1))   # 상위 3개 비중
print(round(summary.loc["FOOD", "count"] / len(df) * 100, 1))                    # 식비 건수 비중
print(round(len(big) / len(df) * 100, 1))                                        # 고액 건수 비중
print(round(big["amount"].sum() / df["amount"].sum() * 100, 1))                  # 고액 금액 비중
big_rate = (big_cat["count"] / summary["count"] * 100).round(1)                  # 카테고리별 고액 비율
print(big_rate.sort_values(ascending=False))

# [핵심 인사이트]
# 지출은 상위 3개 카테고리에 몰린다. 장보기(21.3%), 쇼핑(19.3%), 식비(18.0%)가 전체 지출 약 5,466만 원의 58.7%를 차지한다. 식비는 건수가 324건(27.0%)으로 가장 많지만 건당 평균이 약 30,403원이라 금액 순위는 3위다.
# 고액 거래(15만 원 초과)는 건수는 적지만 금액 비중이 크다. 55건으로 전체의 4.6%지만 금액으로는 18,028,000원, 전체의 33.0%를 차지한다. 고액 거래는 식비 13건(23.2%)이 가장 많고, 구독 10건(16.3%)이 뒤를 잇는다.
# 시기와 카테고리별 고액 비율에 차이가 있다. 고액 거래는 3월에 2건뿐이고 나머지 달은 7~13건이다. 카테고리 안에서 고액 거래가 차지하는 비율은 구독이 11.6%(86건 중 10건)로 가장 높다.